# Welcome, and a price you can't trust

**Lecture 1 · Build** · Géron, Chapters 1–2

Applications of Machine Learning — BSc Mathematics of Artificial Intelligence

---

**How to use this notebook.** You are not expected to type the code. You are
expected to *read* it before you run it, and to be able to say what every line
does and what would break if it changed. Cells marked **⚠ read before running**
contain a defect on purpose.

Run the cells in order. Anything that takes more than a few seconds says so.

## 1 · Setup

In [ ]:
# --- setup -------------------------------------------------------------------
# Not examinable: this is engineering hygiene, not machine learning. It is here
# because a version mismatch produces a confusing error twenty cells later.
import sys, sklearn, numpy as np, pandas as pd, matplotlib

print(f"python       {sys.version.split()[0]}")
print(f"scikit-learn {sklearn.__version__}")
print(f"numpy        {np.__version__}")
print(f"pandas       {pd.__version__}")

# root_mean_squared_error arrived in scikit-learn 1.4
assert tuple(int(p) for p in sklearn.__version__.split(".")[:2]) >= (1, 4), \
    "This notebook needs scikit-learn >= 1.4.  In Colab: %pip install -U scikit-learn"

RANDOM_STATE = 42          # every split, every model, every shuffle
pd.set_option("display.width", 100)

## 2 · The data

In [ ]:
# --- the data ----------------------------------------------------------------
# A function, not a manual download: the data will change, and you will need
# this on another machine.  ~5 s the first time, instant afterwards.
from pathlib import Path
import tarfile, urllib.request

def load_housing():
    tarball = Path("datasets/housing.tgz")
    if not tarball.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball)
        with tarfile.open(tarball) as t:
            t.extractall(path="datasets", filter="data")
    return pd.read_csv("datasets/housing/housing.csv")

housing_full = load_housing()

assert housing_full.shape == (20640, 10), f"unexpected shape {housing_full.shape}"
print(f"{len(housing_full):,} districts, {housing_full.shape[1]} columns")
housing_full.head()

### What is in it

Ten attributes per district. One of them is not numeric, and one column has
holes in it. Find both before reading on.

In [ ]:
housing_full.info()

In [ ]:
n_missing = housing_full["total_bedrooms"].isna().sum()
print(f"total_bedrooms is missing in {n_missing} districts "
      f"({100 * n_missing / len(housing_full):.1f}%)")
print()
print(housing_full["ocean_proximity"].value_counts())

`ISLAND` has five districts in the whole of California. Remember that; it comes
back in the next lecture and it does not announce itself when it breaks.

## 3 · Split before you look

This is the first rule and the easiest one to break. Everything you learn from
the data *before* the split leaks into the choices you make afterwards — through
you, not through the code. There is no library that prevents this.

We stratify on income because the experts told us income predicts price. A
random split gets the income mix wrong by up to 6.4%; stratifying gets it wrong
by 0.36%.

In [ ]:
from sklearn.model_selection import train_test_split

income_cat = pd.cut(housing_full["median_income"],
                    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                    labels=[1, 2, 3, 4, 5])

train_set, test_set = train_test_split(
    housing_full, test_size=0.2, random_state=RANDOM_STATE, stratify=income_cat)

# assert, do not hope
assert len(train_set) + len(test_set) == len(housing_full)
assert set(train_set.index).isdisjoint(test_set.index), "the split overlaps"
print(f"train {len(train_set):,}   test {len(test_set):,}")

# From here to the very last cell, `test_set` is not touched again.
housing = train_set.copy()

## 4 · Look — at the training set only

Two things should jump out of the histograms. Take thirty seconds before you
scroll.

In [ ]:
import matplotlib.pyplot as plt

housing.hist(bins=50, figsize=(12, 8))
plt.tight_layout(); plt.show()

**The income is not in dollars** — it is scaled, and capped at 15.0001.

**The target is capped too**, and the target is our label. Count it rather than
squinting at it:

In [ ]:
capped = (housing["median_house_value"] >= 500_000).sum()
print(f"{capped} districts sit at the cap "
      f"({100 * capped / len(housing):.1f}% of the training set)")

# which values do districts actually pile up on?
counts = housing["median_house_value"].value_counts()
print(f"\na typical price is shared by {counts.median():.0f} districts")
print("\nthe five commonest values below the cap:")
print(counts.drop(counts.index.max()).head(5))

Every one of those is a multiple of **$12,500**. They are artefacts of how the
survey recorded prices, not facts about California.

A well-known description of this dataset names fainter lines at \$450,000,
\$350,000 and \$280,000. Check that claim against the counts above before you
believe it — one of the three is real, one is marginal, and one is
indistinguishable from the background.

In [ ]:
for value in (450_000, 350_000, 280_000):
    print(f"${value:>9,}  {counts.get(value, 0):>4d} districts")

## 5 · A number to compare against

Rule 2 of this course: *a metric with nothing to compare it to is decoration.*

So before building anything, measure the dumbest possible model — predict the
same number for every district. Everything you build today has to beat this, and
by how much is the only thing that will make your RMSE mean anything.

In [ ]:
from sklearn.metrics import root_mean_squared_error

y_train = housing["median_house_value"]
y_test  = test_set["median_house_value"]

baseline = np.full(len(y_test), y_train.mean())
baseline_rmse = root_mean_squared_error(y_test, baseline)
print(f"predict the training mean  ->  RMSE ${baseline_rmse:,.0f}")
print(f"the human experts are off by about 30%, i.e. roughly  ${0.30 * 200_000:,.0f}")

## 6 · Commit

**Stop. On paper, now.** Not in this notebook — on paper, where you cannot
quietly revise it.

```
Metric:                                        ____________
Target RMSE for a good system:               $ ____________
RMSE I expect from the model I build today:  $ ____________
```

A prediction you can silently revise is not a prediction.

## 7 · An assistant writes the preprocessing

Here is a real request and the code it returns. **⚠ Read before running.** It
runs, it imports nothing exotic, and it prints a believable number.

> *"Load the housing data, scale the features and split it into training and
> test sets."*

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

X_all = housing_full.select_dtypes(include=[np.number]).drop(
    columns=["median_house_value"])
y_all = housing_full["median_house_value"]

prep = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
X_scaled = prep.fit_transform(X_all)          # <-- all 20,640 rows

X_tr, X_te, y_tr, y_te = train_test_split(
    X_scaled, y_all, test_size=0.2, random_state=RANDOM_STATE)

leaky = LinearRegression().fit(X_tr, y_tr)
print(f"RMSE ${root_mean_squared_error(y_te, leaky.predict(X_te)):,.0f}   looks fine")

### Reviewer question 1: what touched the test set?

`fit_transform` ran on **all** the rows. The median that fills the missing
values, and the mean and standard deviation that scale every column, were all
computed from a set that includes the rows we then call the test set.

So the model is evaluated on rows whose own values helped define the
transformation applied to them.

**Now measure the damage** — do not guess:

In [ ]:
# the same thing, done correctly: split first, fit the preprocessing on train
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    X_all, y_all, test_size=0.2, random_state=RANDOM_STATE)

prep_ok = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
honest = LinearRegression().fit(prep_ok.fit_transform(Xc_tr), yc_tr)
honest_rmse = root_mean_squared_error(yc_te, honest.predict(prep_ok.transform(Xc_te)))
leaky_rmse  = root_mean_squared_error(y_te, leaky.predict(X_te))

print(f"leaky   ${leaky_rmse:,.2f}")
print(f"correct ${honest_rmse:,.2f}")
print(f"the leak is worth ${abs(honest_rmse - leaky_rmse):,.2f}")

### About a dollar. So why is it a bug?

Three reasons, and the third is the one that matters:

1. **You did not know it was a dollar until you measured.** Nothing in the code
   said so, and neither did the output.
2. **It is this small for three specific reasons** — centring and scaling is an
   invertible affine map, ordinary least squares is equivariant under one, and
   with 20,640 rows the training and test statistics nearly coincide. Remove any
   one of those and the leak has teeth.
3. **A leaked score and an honest score can be identical**, so you cannot detect
   it from the number. That is why the rule is procedural: *split first* — not
   because the damage is always large, but because you cannot tell whether it is.

You will meet the same error worth far more than a dollar in about an hour.

## 8 · Build it properly

One `Pipeline`, so that cross-validation refits *all* of it on each fold and the
leak becomes structurally impossible rather than merely avoided.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

X_train = housing.drop(columns=["median_house_value"])

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = ["ocean_proximity"]

preprocessing = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

assert set(num_cols) | set(cat_cols) == set(X_train.columns), "a column was dropped"
print(f"{len(num_cols)} numeric + {len(cat_cols)} categorical")

In [ ]:
# ~20 s: the forest is 100 trees on 16,512 rows.
models = {
    "Linear regression": LinearRegression(),
    "Decision tree":     DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random forest":     RandomForestRegressor(n_estimators=100,
                                               random_state=RANDOM_STATE, n_jobs=-1),
}

for name, model in models.items():
    pipe = Pipeline([("prep", preprocessing), ("model", model)]).fit(X_train, y_train)
    rmse = root_mean_squared_error(y_train, pipe.predict(X_train))
    print(f"{name:20s} RMSE on training data  ${rmse:>10,.0f}")

## 9 · Where we are

Three numbers. One of them is zero.

Write your **best RMSE** on the same sheet of paper, next to what you predicted.
Bring it to the next lecture — we open by comparing them, and two of these
numbers are meaningless.

Do not fix anything yet. Being wrong is the point, and the diagnosis is the next
ninety minutes.